In [1]:
import copy
import math
import os
import re
import sys
import time
from collections import defaultdict
from glob import glob

import numpy as np
import pandas as pd
import xarray as xr
import yaml
import zarr

In [2]:
ds_static = xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/static/static.zarr')
lon = ds_static['lon'].values
lat = ds_static['lat'].values

dict_loc = {
    'Pituffik': (76.4, -68.575),
    'Fairbanks': (64.75, -147.4),
    'Guam': (13.475, 144.75),
    'Yuma_PG': (33.125, -114.125),
    'Fort_Bragg': (35.05, -79.115),
}

def time_to_lead_and_stack(ds_list, new_dim="member", labels=None, ref="first"):
    """
    ds_list: list[xr.Dataset], each has a 'time' coord with daily values
    new_dim: name of the new dimension to stack on
    labels: optional labels for new_dim (len == len(ds_list))
    ref: "first" (per-dataset first time) or a numpy/pandas datetime-like scalar
    """
    out = []
    for i, ds in enumerate(ds_list):
        ds = ds.sortby("time")

        if ref == "first":
            t0 = ds["time"].isel(time=0)
        else:
            # global reference (same for all ds)
            t0 = xr.DataArray(ref)

        lead = (ds["time"] - t0).astype("timedelta64[D]")  # daily deltas
        ds2 = ds.assign_coords(lead_time=("time", lead.data)).swap_dims({"time": "lead_time"})
        ds2 = ds2.drop_vars("time")  # optional: remove original coordinate

        out.append(ds2)

    if labels is None:
        labels = list(range(len(out)))

    stacked = xr.concat(out, dim=xr.IndexVariable(new_dim, labels))
    return stacked

In [3]:
# ============================================================================= #
# Gather CESM-SMYLE daily data
# ============================================================================= #

years = np.arange(2001, 2020, 1)
varnames = ['FLDS', 'FSDS', 'PRECSC', 'PRECSL', 'PRECT', 'PSL', 'QREFHT', 'TMQ', 'TREFHT', 'TREFHTMN', 'TREFHTMX', 'U10']

for year_init in years:

    year_start = year_init + 1
    year_end = year_init + 10

    fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/SMYLE_{year_init}-11-01_daily_ensemble.zarr'
    ds = xr.open_zarr(fn)[varnames]
    ds = ds.sel(time=slice(f'{year_start}-01-01T00', f'{year_end}-12-31T00'))

    ds = ds.assign_coords(lon = (((ds.lon + 180) % 360) - 180))
    ds = ds.sortby("lon")
    
    for stn, (lat_mid, lon_mid) in dict_loc.items():
        
        subset = ds.sel(lat=lat_mid, lon=lon_mid, method='nearest')
        
        L = len(subset['time'])
        subset = subset.chunk({'time': L})
        
        save_name = f'/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/{stn}_{year_init}_member.zarr'        
        subset.to_zarr(save_name, mode='w', consolidated=True, compute=True)
        print(save_name)

/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/Pituffik_2001_member.zarr
/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/Fairbanks_2001_member.zarr
/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/Guam_2001_member.zarr
/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/Yuma_PG_2001_member.zarr
/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/Fort_Bragg_2001_member.zarr
/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/Pituffik_2002_member.zarr
/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/Fairbanks_2002_member.zarr
/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/Guam_2002_member.zarr
/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/Yuma_PG_2002_member.zarr
/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/Fort_Bragg_2002_member.zarr
/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/Pituffik_2003_member.zarr
/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/Fairbanks_2003_member.zarr
/glade/campaign/ral/hap/ksha/EPRI_data/CESM

In [9]:
# ============================================================================= #
# Convert CESM-SMYLE daily to annual metrics
# ============================================================================= #

station_names = ['Pituffik', 'Fairbanks', 'Guam', 'Yuma_PG' ,'Fort_Bragg'] # 

for stn in station_names:
    
    t0 = time.perf_counter()
    
    base_dir = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS_STN/{stn}/'
    
    # ========================== #
    # get data
    list_ds = []
    for year in range(1958, 2020):
        fn = f'/glade/campaign/ral/hap/ksha/EPRI_data/CESM_SMYLE_STN/{stn}_{year}_member.zarr'
        ds = xr.open_zarr(fn)[['PRECT', 'TREFHT', 'TREFHTMX', 'TREFHTMN']]
        list_ds.append(ds)
        
    ds_all = time_to_lead_and_stack(list_ds, new_dim="init_time")
    
    ds_all['PRECT'] = ds_all['PRECT'] * 60*60*24 * 1000 # mm per day
    
    ds_all = ds_all.assign_coords({'init_time': np.arange(1959, 2021)})
    lead_year = (ds_all["lead_time"] / np.timedelta64(365, "D")).astype(int)
    ds_all = ds_all.assign_coords(lead_year=("lead_time", lead_year.data))
    
    # ======================= #
    # metrics
    ds_group = ds_all.groupby("lead_year")
    ds_max  = ds_group.max(dim="lead_time",  skipna=True)
    ds_min  = ds_group.min(dim="lead_time",  skipna=True)
    ds_mean  = ds_group.mean(dim="lead_time",  skipna=True)
    ds_30d = ds_group.map(
        lambda x: x.rolling(lead_time=30, min_periods=30).mean().max(dim="lead_time", skipna=True)
    )
    ds_min = ds_min.rename({'TREFHTMN': 'TREFHTMN_min', 'TREFHT': 'TREFHT_min'})[['TREFHTMN_min', 'TREFHT_min']]
    ds_max = ds_max.rename({'PRECT': 'PRECT_max', 'TREFHTMX': 'TREFHTMX_max', 'TREFHT': 'TREFHT_max'})[['PRECT_max', 'TREFHTMX_max', 'TREFHT_max']]
    ds_30d = ds_30d.rename({'TREFHT': 'TREFHT_30d', 'PRECT': 'PRECT_30d'})[['TREFHT_30d', 'PRECT_30d']]
    ds_mean = ds_mean.rename({'PRECT': 'PRECT_mean', 'TREFHT': 'TREFHT_mean'})[['PRECT_mean', 'TREFHT_mean']]
    ds_metrics = xr.merge([ds_min, ds_max, ds_30d, ds_mean])
    ds_metrics = ds_metrics.rename({v: f"{v}_default" for v in ds_metrics.data_vars})
    
    # ========================== #
    # save
    ds_final = ds_metrics
    save_name = base_dir + 'CESM_member_metrics.zarr'
    ds_final.to_zarr(save_name, mode='w')
    print(save_name)
    
    t1 = time.perf_counter()
    print(f"Elapsed: {t1 - t0:.6f} s")

In [5]:
# ============================================================================= #
# Save CESM metrics as netCDF
# ============================================================================= #

def detrend_linear(da, dim="time"):
    """
    Remove a best-fit linear trend along `dim` for each grid point.
    Uses an index-based time axis (0..N-1) to avoid datetime scaling issues.
    """
    t = xr.DataArray(np.arange(da.sizes[dim]), dims=dim, coords={dim: da[dim]})

    valid = np.isfinite(da)
    t_valid = t.where(valid)
    da_valid = da.where(valid)

    t_mean = t_valid.mean(dim, skipna=True)
    y_mean = da_valid.mean(dim, skipna=True)

    cov = ((t_valid - t_mean) * (da_valid - y_mean)).mean(dim, skipna=True)
    var = ((t_valid - t_mean) ** 2).mean(dim, skipna=True)

    slope = cov / var
    intercept = y_mean - slope * t_mean

    trend = slope * t + intercept
    return da - trend
    

dict_loc = {
    'Pituffik': (76.4, -68.575),
    'Fairbanks': (64.75, -147.4),
    'Guam': (13.475, 144.75),
    'Yuma_PG': (33.125, -114.125),
    'Fort_Bragg': (35.05, -79.115),
}
keys = list(dict_loc.keys())

base_dir = '/glade/derecho/scratch/ksha/EPRI_data/METRICS_STN/'

lat = [coords[0] for coords in dict_loc.values()]
lon = [coords[1] for coords in dict_loc.values()]

list_ds = []
for stn in keys:
    fn_CESM = base_dir + f'{stn}/CESM_member_metrics.zarr'
    ds_CESM = xr.open_zarr(fn_CESM)
    ds_CESM = ds_CESM.rename({'init_time': 'initialization', 'lead_year': 'lead_time'})
    ds_CESM = ds_CESM.drop_vars(['lat', 'lon'])
    list_ds.append(ds_CESM)
    
ds_CESM_all = xr.concat(list_ds, dim='site')
ds_CESM_all = ds_CESM_all.assign_coords({'site': keys})
ds_CESM_all = ds_CESM_all.assign_coords(lat=('lat', lat))
ds_CESM_all = ds_CESM_all.assign_coords(lon=('lon', lon))

# ================================================ #
# anomaly
ds_CESM_all_anom = ds_CESM_all.copy()
vars_ = list(ds_CESM_all.keys())

for v in vars_:
    clim = ds_CESM_all[v].mean("initialization")
    ds_CESM_all_anom[v] = ds_CESM_all[v] - clim
ds_CESM_all_anom = ds_CESM_all_anom[vars_]
ds_CESM_all_anom = ds_CESM_all_anom.rename({v: v[:-7]+'anom' for v in ds_CESM_all_anom.data_vars})

# ================================================ #
# detrend
ds_CESM_all_detrend = ds_CESM_all.copy()
for v in vars_:
    ds_CESM_all_detrend[v] = detrend_linear(ds_CESM_all[v], dim="initialization")
ds_CESM_all_detrend = ds_CESM_all_detrend[vars_]
ds_CESM_all_detrend = ds_CESM_all_detrend.rename({v: v[:-7]+'detrend' for v in ds_CESM_all_detrend.data_vars})

ds_merge = xr.merge([ds_CESM_all, ds_CESM_all_anom, ds_CESM_all_detrend])
ds_merge = ds_merge.rename({'initialization': 'gen_date'})
ds_merge = ds_merge.load()

# ================================================ #
# NINO 3.4
fn = '/glade/campaign/ral/hap/ksha/EPRI_data/CESM_OCN/ENSO_index.npy'
NINO34 = np.load(fn, allow_pickle=True)[()]

years = np.array(list(NINO34.keys()))

nino_index = np.empty((5, 10, 62))

for i_year, year in enumerate(years):
    nino_index[:, :, i_year] = NINO34[int(year)][2:][None, :]

ds_merge["nino_34"] = xr.DataArray(
    nino_index,
    dims=("site", "lead_time", "gen_date"),
    coords={
        "site": ds_merge .site,
        "lead_time": ds_merge .lead_time,
        "gen_date": ds_merge .gen_date,
    },)

# ================================================ #
# NAO
ds_NAO = xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/METRICS/NAO.zarr')
ds_NAO = ds_NAO.load()
ds_NAO = ds_NAO.rename({'lead_year': 'lead_time', 'init_year': 'gen_date'})
ds_NAO = ds_NAO.rename({'nao_ann_eof': 'NAO_annual_mean', 'nao_djf_eof': 'NAO_DJF'})[['NAO_annual_mean', 'NAO_DJF']]
ds_NAO = ds_NAO.expand_dims({'site': 5})
ds_NAO = ds_NAO.drop_vars(('mode',))
ds_NAO = ds_NAO.assign_coords({'gen_date': ds_merge['gen_date']})
ds_merge = xr.merge([ds_merge, ds_NAO])

# ================================================ #
# GBI
ds_GBI = xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/METRICS/GBI.zarr')
ds_GBI = ds_GBI.load()
ds_GBI = ds_GBI.rename({'init_year': 'gen_date', 'lead_year': 'lead_time'})
ds_GBI = ds_GBI.expand_dims({'site': 5})
ds_GBI = ds_GBI.assign_coords({'gen_date': ds_merge['gen_date']})
ds_merge = xr.merge([ds_merge, ds_GBI])

ds_merge['GBI_30d'] = ds_merge['GBI_30d'].transpose('site', 'lead_time', 'gen_date')
ds_merge['GBI_mean'] = ds_merge['GBI_mean'].transpose('site', 'lead_time', 'gen_date')

rename_dict = {
    'PRECT_30d_default'   : 'CESM_precip_annual_max_30d',
    'PRECT_max_default'   : 'CESM_precip_max_daily',
    'PRECT_mean_default'  : 'CESM_precip_mean',
    'TREFHTMN_min_default': 'CESM_t2m_min_hour',
    'TREFHTMX_max_default': 'CESM_t2m_max_hour',
    'TREFHT_30d_default'  : 'CESM_t2m_annual_max_30d',
    'TREFHT_max_default'  : 'CESM_t2m_max_daily',
    'TREFHT_mean_default' : 'CESM_t2m_mean',
    'TREFHT_min_default'  : 'CESM_t2m_min_daily',
    'PRECT_30d_anom'      : 'CESM_anomaly_precip_annual_max_30d',
    'PRECT_max_anom'      : 'CESM_anomaly_precip_max_daily',
    'PRECT_mean_anom'     : 'CESM_anomaly_precip_mean',
    'TREFHTMN_min_anom'   : 'CESM_anomaly_t2m_min_hour',
    'TREFHTMX_max_anom'   : 'CESM_anomaly_t2m_max_hour',
    'TREFHT_30d_anom'     : 'CESM_anomaly_t2m_annual_max_30d',
    'TREFHT_max_anom'     : 'CESM_anomaly_t2m_max_daily',
    'TREFHT_mean_anom'    : 'CESM_anomaly_t2m_mean',
    'TREFHT_min_anom'     : 'CESM_anomaly_t2m_min_daily',
    'PRECT_30d_detrend'   : 'CESM_detrend_precip_annual_max_30d',
    'PRECT_max_detrend'   : 'CESM_detrend_precip_max_daily',
    'PRECT_mean_detrend'  : 'CESM_detrend_precip_mean',
    'TREFHTMN_min_detrend': 'CESM_detrend_t2m_min_hour',
    'TREFHTMX_max_detrend': 'CESM_detrend_t2m_max_hour',
    'TREFHT_30d_detrend'  : 'CESM_detrend_t2m_annual_max_30d',
    'TREFHT_max_detrend'  : 'CESM_detrend_t2m_max_daily',
    'TREFHT_mean_detrend' : 'CESM_detrend_t2m_mean',
    'TREFHT_min_detrend'  : 'CESM_detrend_t2m_min_daily',
    'nino_34'             : 'nino_34'
}

ds_merge = ds_merge.rename(rename_dict)
ds_merge.to_netcdf(
    '/glade/derecho/scratch/ksha/EPRI_AnEn/baseline_CESM_member_20260601.nc',
    format="NETCDF4_CLASSIC",
    engine="netcdf4",
    mode='w'
)

### QC

In [6]:
ds_old = xr.open_dataset('/glade/derecho/scratch/ksha/EPRI_AnEn/baseline_CESM_member_20260409.nc')

In [11]:
test_old = ds_old['CESM_precip_annual_max_30d'].values

In [12]:
test_new = ds_merge['CESM_precip_annual_max_30d'].values

In [16]:
np.sum(np.abs(test_old - test_new))

0.0